# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정
9. 보고서용 실험 실행
10. 하이퍼파라미터 비교 실험
11. 전체 테스트와 제출 전 확인

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.


## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 보고서용 실험 실행

아래 셀들은 `REPORT.md`에 적을 BPE, 사전 학습, 미세 조정 결과를 수집합니다. Colab GPU에서 실행하는 것을 기준으로 하며, 테스트는 실행하지 않습니다.


In [ ]:
# 보고서용 실험 설정: Basic 기본값
import json
import random
import time
import platform
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from bpe import BPETokenizer
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, evaluate_model, generate
from finetune import (
    ReviewSentimentDataset,
    GPTForSequenceClassification,
    train_epoch_sentiment,
    evaluate_sentiment,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPORT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", REPORT_DEVICE)
if REPORT_DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

REPORT_PATHS = {
    "vocab": repo_dir / "data" / "vocab_bpe_basic_3000.json",
    "checkpoint": repo_dir / "checkpoints" / "report_basic_last.pt",
    "sentiment_train": repo_dir / "data" / "nsmc_sentiment_train.jsonl",
    "sentiment_val": repo_dir / "data" / "nsmc_sentiment_val.jsonl",
    "sentiment_test": repo_dir / "data" / "nsmc_sentiment_test.jsonl",
}

REPORT_BPE = {
    "vocab_size": 3000,
    "corpus_limit": 1_500_000,
}

REPORT_MODEL_CONFIG = {
    "vocab_size": REPORT_BPE["vocab_size"],
    "context_length": 128,
    "emb_dim": 192,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

REPORT_PRETRAIN = {
    "batch_size": 8,
    "num_epochs": 1,
    "lr": 3e-4,
    "weight_decay": 0.01,
    "eval_freq": 100,
    "eval_iter": 20,
    "start_context": "이 영화는",
    "max_new_tokens": 50,
    "temperature": 0.8,
    "top_k": 40,
}

REPORT_FINETUNE = {
    "max_length": 128,
    "batch_size": 16,
    "num_epochs": 1,
    "backbone_lr": 1e-4,
    "classifier_lr": 3e-4,
    "drop_rate": 0.1,
}


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def fmt_seconds(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}초"
    return f"{seconds / 60:.1f}분"


def compact_text(text: str, max_len: int = 180) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def decode_token_ids_safe(tokenizer, token_ids: list[int]) -> str:
    """생성된 token ID를 UTF-8 오류로 중단되지 않게 문자열로 바꿉니다."""
    try:
        return tokenizer.decode(token_ids)
    except UnicodeDecodeError:
        special_ids = {
            tokenizer.get_pad_id(),
            tokenizer.get_unk_id(),
            tokenizer.get_bos_id(),
            tokenizer.get_eos_id(),
        }
        byte_values = []
        for token_id in token_ids:
            if token_id in special_ids:
                continue
            try:
                byte_values.extend(tokenizer.token_to_bytes(token_id))
            except (KeyError, ValueError):
                continue
        return bytes(byte_values).decode("utf-8", errors="replace")

for name, value in REPORT_PATHS.items():
    print(f"{name}: {value}")


In [ ]:
# BPE vocabulary 학습 또는 로드
if not corpus:
    raise RuntimeError("LM train corpus가 비어 있습니다. 먼저 NSMC 데이터 준비 셀을 실행하세요.")

bpe_corpus = corpus[: REPORT_BPE["corpus_limit"]]
report_tokenizer = BPETokenizer(vocab_size=REPORT_BPE["vocab_size"])

bpe_started = time.perf_counter()
if REPORT_PATHS["vocab"].exists():
    report_tokenizer.load(REPORT_PATHS["vocab"])
    report_bpe_mode = "loaded"
else:
    REPORT_PATHS["vocab"].parent.mkdir(parents=True, exist_ok=True)
    report_tokenizer.train(bpe_corpus)
    report_tokenizer.save(REPORT_PATHS["vocab"])
    report_bpe_mode = "trained"
report_bpe_elapsed = time.perf_counter() - bpe_started

restore_samples = [
    "이 영화는 정말 좋았다!",
    "연기, 음악, 연출 모두 좋음 ㅋㅋ",
    "한글 English 123!?",
]
report_bpe_restore = []
for text in restore_samples:
    decoded = report_tokenizer.decode(report_tokenizer.encode(text, add_bos_eos=True))
    report_bpe_restore.append({"text": text, "decoded": decoded, "ok": decoded == text})

print("BPE mode:", report_bpe_mode)
print("corpus chars:", len(bpe_corpus))
print("target vocab_size:", REPORT_BPE["vocab_size"])
print("actual vocab size:", len(report_tokenizer.id_to_token))
print("elapsed:", fmt_seconds(report_bpe_elapsed))
print("vocab path:", REPORT_PATHS["vocab"])
for row in report_bpe_restore:
    print(f"restore ok={row['ok']}: {row['decoded']}")


In [ ]:
# 사전 학습 준비: tokenization, dataloader, model, optimizer
lm_train_text = corpus[: REPORT_BPE["corpus_limit"]]
lm_val_text = val_corpus
if not lm_train_text or not lm_val_text:
    raise RuntimeError("LM train/val corpus가 필요합니다. 데이터 준비 셀을 먼저 실행하세요.")

tokenize_started = time.perf_counter()
report_train_token_ids = report_tokenizer.encode(lm_train_text)
report_val_token_ids = report_tokenizer.encode(lm_val_text)
report_tokenize_elapsed = time.perf_counter() - tokenize_started

report_train_loader = create_dataloader(
    report_train_token_ids,
    context_length=REPORT_MODEL_CONFIG["context_length"],
    batch_size=REPORT_PRETRAIN["batch_size"],
    stride=REPORT_MODEL_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)
report_val_loader = create_dataloader(
    report_val_token_ids,
    context_length=REPORT_MODEL_CONFIG["context_length"],
    batch_size=REPORT_PRETRAIN["batch_size"],
    stride=REPORT_MODEL_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

if len(report_train_loader) == 0 or len(report_val_loader) == 0:
    raise RuntimeError("사전 학습 DataLoader가 비어 있습니다. corpus 크기와 context_length를 확인하세요.")

report_gpt = GPTModel(REPORT_MODEL_CONFIG).to(REPORT_DEVICE)
report_optimizer = torch.optim.AdamW(
    report_gpt.parameters(),
    lr=REPORT_PRETRAIN["lr"],
    weight_decay=REPORT_PRETRAIN["weight_decay"],
)
report_model_param_count = count_parameters(report_gpt)

print("train tokens:", len(report_train_token_ids))
print("val tokens:", len(report_val_token_ids))
print("train batches:", len(report_train_loader))
print("val batches:", len(report_val_loader))
print("tokenization elapsed:", fmt_seconds(report_tokenize_elapsed))
print("model parameters:", f"{report_model_param_count:,}")


In [ ]:
# 사전 학습 실행: train/validation loss history와 생성 샘플 수집
report_pretrain_history = []
report_pretrain_samples = []
report_tokens_seen = 0
report_global_step = 0
pretrain_started = time.perf_counter()

for epoch in range(1, REPORT_PRETRAIN["num_epochs"] + 1):
    report_gpt.train()
    for input_batch, target_batch in report_train_loader:
        report_optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch, report_gpt, REPORT_DEVICE)
        loss.backward()
        report_optimizer.step()

        report_tokens_seen += input_batch.numel()
        report_global_step += 1

        if report_global_step % REPORT_PRETRAIN["eval_freq"] == 0:
            train_loss, val_loss = evaluate_model(
                report_gpt,
                report_train_loader,
                report_val_loader,
                REPORT_DEVICE,
                REPORT_PRETRAIN["eval_iter"],
            )
            row = {
                "epoch": epoch,
                "step": report_global_step,
                "tokens_seen": report_tokens_seen,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "elapsed_sec": time.perf_counter() - pretrain_started,
            }
            report_pretrain_history.append(row)
            print(
                f"epoch={epoch} step={report_global_step} "
                f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                f"elapsed={fmt_seconds(row['elapsed_sec'])}"
            )

    train_loss, val_loss = evaluate_model(
        report_gpt,
        report_train_loader,
        report_val_loader,
        REPORT_DEVICE,
        REPORT_PRETRAIN["eval_iter"],
    )
    row = {
        "epoch": epoch,
        "step": report_global_step,
        "tokens_seen": report_tokens_seen,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "elapsed_sec": time.perf_counter() - pretrain_started,
    }
    report_pretrain_history.append(row)
    print(
        f"epoch_end={epoch} step={report_global_step} "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"elapsed={fmt_seconds(row['elapsed_sec'])}"
    )

    report_gpt.eval()
    start_ids = torch.tensor(
        report_tokenizer.encode(REPORT_PRETRAIN["start_context"]),
        dtype=torch.long,
        device=REPORT_DEVICE,
    ).unsqueeze(0)
    with torch.no_grad():
        sampled_ids = generate(
            model=report_gpt,
            idx=start_ids,
            max_new_tokens=REPORT_PRETRAIN["max_new_tokens"],
            context_size=REPORT_MODEL_CONFIG["context_length"],
            temperature=REPORT_PRETRAIN["temperature"],
            top_k=REPORT_PRETRAIN["top_k"],
            eos_id=report_tokenizer.get_eos_id(),
        )
    sample_text = decode_token_ids_safe(report_tokenizer, sampled_ids.squeeze(0).tolist())
    report_pretrain_samples.append({"epoch": epoch, "text": sample_text})
    print("generated sample:", compact_text(sample_text, max_len=240))

report_pretrain_elapsed = time.perf_counter() - pretrain_started
REPORT_PATHS["checkpoint"].parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": report_gpt.state_dict(),
        "optimizer_state_dict": report_optimizer.state_dict(),
        "config": REPORT_MODEL_CONFIG,
        "history": report_pretrain_history,
        "samples": report_pretrain_samples,
        "tokens_seen": report_tokens_seen,
        "global_step": report_global_step,
    },
    REPORT_PATHS["checkpoint"],
)

print("pretrain elapsed:", fmt_seconds(report_pretrain_elapsed))
print("checkpoint path:", REPORT_PATHS["checkpoint"])


In [ ]:
# 감성 분류 미세 조정: validation/test loss와 accuracy 수집
for split_name in ["sentiment_train", "sentiment_val", "sentiment_test"]:
    if not REPORT_PATHS[split_name].exists():
        raise RuntimeError(f"감성 분류 데이터가 없습니다: {REPORT_PATHS[split_name]}")

sentiment_train_data = read_jsonl(REPORT_PATHS["sentiment_train"])
sentiment_val_data = read_jsonl(REPORT_PATHS["sentiment_val"])
sentiment_test_data = read_jsonl(REPORT_PATHS["sentiment_test"])

sentiment_train_dataset = ReviewSentimentDataset(
    sentiment_train_data,
    tokenizer=report_tokenizer,
    max_length=REPORT_FINETUNE["max_length"],
)
sentiment_val_dataset = ReviewSentimentDataset(
    sentiment_val_data,
    tokenizer=report_tokenizer,
    max_length=REPORT_FINETUNE["max_length"],
)
sentiment_test_dataset = ReviewSentimentDataset(
    sentiment_test_data,
    tokenizer=report_tokenizer,
    max_length=REPORT_FINETUNE["max_length"],
)

sentiment_train_loader = DataLoader(
    sentiment_train_dataset,
    batch_size=REPORT_FINETUNE["batch_size"],
    shuffle=True,
    drop_last=False,
)
sentiment_val_loader = DataLoader(
    sentiment_val_dataset,
    batch_size=REPORT_FINETUNE["batch_size"],
    shuffle=False,
    drop_last=False,
)
sentiment_test_loader = DataLoader(
    sentiment_test_dataset,
    batch_size=REPORT_FINETUNE["batch_size"],
    shuffle=False,
    drop_last=False,
)

report_sentiment_model = GPTForSequenceClassification(
    report_gpt,
    num_labels=2,
    drop_rate=REPORT_FINETUNE["drop_rate"],
).to(REPORT_DEVICE)
report_sentiment_optimizer = torch.optim.AdamW(
    [
        {"params": report_sentiment_model.gpt.parameters(), "lr": REPORT_FINETUNE["backbone_lr"]},
        {"params": report_sentiment_model.classifier.parameters(), "lr": REPORT_FINETUNE["classifier_lr"]},
    ]
)

report_sentiment_history = []
sentiment_started = time.perf_counter()
for epoch in range(1, REPORT_FINETUNE["num_epochs"] + 1):
    train_loss, train_acc = train_epoch_sentiment(
        report_sentiment_model,
        sentiment_train_loader,
        report_sentiment_optimizer,
        REPORT_DEVICE,
    )
    val_loss, val_acc = evaluate_sentiment(
        report_sentiment_model,
        sentiment_val_loader,
        REPORT_DEVICE,
    )
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "elapsed_sec": time.perf_counter() - sentiment_started,
    }
    report_sentiment_history.append(row)
    print(
        f"epoch={epoch} train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
        f"elapsed={fmt_seconds(row['elapsed_sec'])}"
    )

report_sentiment_test_loss, report_sentiment_test_acc = evaluate_sentiment(
    report_sentiment_model,
    sentiment_test_loader,
    REPORT_DEVICE,
)
report_sentiment_elapsed = time.perf_counter() - sentiment_started

print("test_loss:", f"{report_sentiment_test_loss:.4f}")
print("test_acc:", f"{report_sentiment_test_acc:.4f}")
print("sentiment elapsed:", fmt_seconds(report_sentiment_elapsed))
print("dataset sizes:", len(sentiment_train_dataset), len(sentiment_val_dataset), len(sentiment_test_dataset))


In [ ]:
# REPORT.md에 붙여넣기 좋은 Markdown 요약 출력
from IPython.display import Markdown, display

final_pretrain = report_pretrain_history[-1] if report_pretrain_history else None
final_sentiment = report_sentiment_history[-1] if report_sentiment_history else None
sample_text = report_pretrain_samples[-1]["text"] if report_pretrain_samples else ""

def md_table(rows):
    lines = ["| 항목 | 내용 |", "| --- | --- |"]
    for key, value in rows:
        lines.append(f"| {key} | {value} |")
    return "\n".join(lines)

summary_parts = []
summary_parts.append("### BPE")
summary_parts.append(md_table([
    ("BPE 방식", "UTF-8 byte-level BPE"),
    ("vocab_size", REPORT_BPE["vocab_size"]),
    ("실제 vocabulary 크기", len(report_tokenizer.id_to_token)),
    ("학습 corpus 크기", f"corpus[:{REPORT_BPE['corpus_limit']:,}] / {len(bpe_corpus):,} chars"),
    ("어휘 학습/로드 시간", fmt_seconds(report_bpe_elapsed)),
    ("vocabulary 저장 경로", str(REPORT_PATHS["vocab"].relative_to(repo_dir))),
    ("인코딩/디코딩 복원", ", ".join("성공" if row["ok"] else "실패" for row in report_bpe_restore)),
]))

summary_parts.append("\n### 모델 구조")
summary_parts.append(md_table([
    ("전체 구조", "InputEmbedding -> 4 x TransformerBlock -> LayerNorm -> LM head"),
    ("vocab_size", REPORT_MODEL_CONFIG["vocab_size"]),
    ("context_length", REPORT_MODEL_CONFIG["context_length"]),
    ("emb_dim", REPORT_MODEL_CONFIG["emb_dim"]),
    ("n_heads", REPORT_MODEL_CONFIG["n_heads"]),
    ("n_layers", REPORT_MODEL_CONFIG["n_layers"]),
    ("drop_rate", REPORT_MODEL_CONFIG["drop_rate"]),
    ("qkv_bias", REPORT_MODEL_CONFIG["qkv_bias"]),
    ("총 파라미터 수", f"{report_model_param_count:,}"),
]))

summary_parts.append("\n### 사전 학습")
summary_parts.append(md_table([
    ("batch_size", REPORT_PRETRAIN["batch_size"]),
    ("num_epochs", REPORT_PRETRAIN["num_epochs"]),
    ("lr / weight_decay", f"{REPORT_PRETRAIN['lr']} / {REPORT_PRETRAIN['weight_decay']}"),
    ("eval_freq / eval_iter", f"{REPORT_PRETRAIN['eval_freq']} / {REPORT_PRETRAIN['eval_iter']}"),
    ("최종 train loss", f"{final_pretrain['train_loss']:.4f}" if final_pretrain else ""),
    ("최종 validation loss", f"{final_pretrain['val_loss']:.4f}" if final_pretrain else ""),
    ("학습 소요 시간", fmt_seconds(report_pretrain_elapsed)),
    ("checkpoint 경로", str(REPORT_PATHS["checkpoint"].relative_to(repo_dir))),
]))
summary_parts.append("\n생성 샘플:")
summary_parts.append("```text\n" + compact_text(sample_text, max_len=600) + "\n```")

summary_parts.append("\n### 미세 조정")
summary_parts.append(md_table([
    ("과제", "NSMC 리뷰 긍정/부정 분류"),
    ("max_length", REPORT_FINETUNE["max_length"]),
    ("batch_size", REPORT_FINETUNE["batch_size"]),
    ("num_epochs", REPORT_FINETUNE["num_epochs"]),
    ("backbone learning rate", REPORT_FINETUNE["backbone_lr"]),
    ("classifier learning rate", REPORT_FINETUNE["classifier_lr"]),
    ("validation loss / accuracy", f"{final_sentiment['val_loss']:.4f} / {final_sentiment['val_acc']:.4f}" if final_sentiment else ""),
    ("test loss / accuracy", f"{report_sentiment_test_loss:.4f} / {report_sentiment_test_acc:.4f}"),
    ("학습 소요 시간", fmt_seconds(report_sentiment_elapsed)),
]))

summary_parts.append("\n### 실험 환경")
summary_parts.append(md_table([
    ("Python", platform.python_version()),
    ("PyTorch", torch.__version__),
    ("실행 환경", "Colab GPU" if REPORT_DEVICE.type == "cuda" else "CPU"),
    ("GPU/CPU 정보", torch.cuda.get_device_name(0) if REPORT_DEVICE.type == "cuda" else platform.processor()),
    ("총 학습 소요 시간", fmt_seconds(report_pretrain_elapsed + report_sentiment_elapsed)),
]))

report_markdown = "\n\n".join(summary_parts)
display(Markdown(report_markdown))
print(report_markdown)


## 10. 하이퍼파라미터 비교 실험

아래 셀은 여러 config를 순서대로 학습해 train/validation loss를 비교합니다. 전체 grid는 648개 조합이므로 기본값은 대표 조합만 실행하고, 필요하면 `SWEEP_MODE`, `SWEEP_MAX_TRIALS`, `MANUAL_SWEEP_CONFIGS`를 수정해 실험 범위를 조절하세요.


In [ ]:
# 하이퍼파라미터 sweep 설정
import gc
import itertools

if "report_tokenizer" not in globals():
    raise RuntimeError("먼저 'BPE vocabulary 학습 또는 로드' 셀을 실행해 report_tokenizer를 준비하세요.")

HYPERPARAMETER_GRID = {
    "batch_size": [2, 4, 8, 16],
    "drop_rate": [0.0, 0.1, 0.2],
    "learning_rate": [1e-4, 3e-4, 5e-4],
    "context_length": [64, 128],
    "n_layers": [1, 2, 4],
    "emb_dim": [64, 128, 192],
}

BASE_SWEEP_CONFIG = {
    "batch_size": 8,
    "drop_rate": 0.1,
    "learning_rate": 3e-4,
    "context_length": 128,
    "n_layers": 4,
    "emb_dim": 192,
}

# manual: 아래 대표 조합만 실행합니다. grid: 전체 grid에서 SWEEP_MAX_TRIALS개만 실행합니다.
SWEEP_MODE = "manual"
SWEEP_MAX_TRIALS = 10
SWEEP_CORPUS_LIMIT = 300_000
SWEEP_VAL_CORPUS_LIMIT = 80_000
SWEEP_MAX_STEPS = 200
SWEEP_EVAL_FREQ = 50
SWEEP_EVAL_ITER = 10
SWEEP_WEIGHT_DECAY = 0.01
SWEEP_START_CONTEXT = "이 영화는"
SWEEP_MAX_NEW_TOKENS = 30

MANUAL_SWEEP_CONFIGS = [
    {**BASE_SWEEP_CONFIG, "name": "base"},
    {**BASE_SWEEP_CONFIG, "batch_size": 4, "name": "batch_size=4"},
    {**BASE_SWEEP_CONFIG, "batch_size": 16, "name": "batch_size=16"},
    {**BASE_SWEEP_CONFIG, "drop_rate": 0.0, "name": "drop_rate=0.0"},
    {**BASE_SWEEP_CONFIG, "drop_rate": 0.2, "name": "drop_rate=0.2"},
    {**BASE_SWEEP_CONFIG, "learning_rate": 1e-4, "name": "lr=1e-4"},
    {**BASE_SWEEP_CONFIG, "learning_rate": 5e-4, "name": "lr=5e-4"},
    {**BASE_SWEEP_CONFIG, "context_length": 64, "name": "context_length=64"},
    {**BASE_SWEEP_CONFIG, "n_layers": 2, "name": "n_layers=2"},
    {**BASE_SWEEP_CONFIG, "emb_dim": 128, "name": "emb_dim=128"},
]


def make_grid_configs(grid: dict, max_trials: int | None = None) -> list[dict]:
    keys = list(grid.keys())
    configs = [dict(zip(keys, values)) for values in itertools.product(*(grid[key] for key in keys))]
    for idx, cfg in enumerate(configs, start=1):
        cfg["name"] = f"grid_{idx:03d}"
    return configs if max_trials is None else configs[:max_trials]


def get_sweep_configs() -> list[dict]:
    if SWEEP_MODE == "manual":
        return MANUAL_SWEEP_CONFIGS[:SWEEP_MAX_TRIALS]
    if SWEEP_MODE == "grid":
        return make_grid_configs(HYPERPARAMETER_GRID, max_trials=SWEEP_MAX_TRIALS)
    raise ValueError("SWEEP_MODE는 'manual' 또는 'grid'만 사용할 수 있습니다.")


def build_sweep_model_config(cfg: dict) -> dict:
    emb_dim = int(cfg["emb_dim"])
    n_heads = 4
    if emb_dim % n_heads != 0:
        raise ValueError(f"emb_dim={emb_dim}은 n_heads={n_heads}로 나누어 떨어져야 합니다.")
    return {
        "vocab_size": REPORT_BPE["vocab_size"],
        "context_length": int(cfg["context_length"]),
        "emb_dim": emb_dim,
        "n_heads": n_heads,
        "n_layers": int(cfg["n_layers"]),
        "drop_rate": float(cfg["drop_rate"]),
        "qkv_bias": False,
    }


def safe_decode_token_ids(tokenizer, token_ids: list[int]) -> str:
    return decode_token_ids_safe(tokenizer, token_ids)

sweep_configs = get_sweep_configs()
print("sweep mode:", SWEEP_MODE)
print("trial count:", len(sweep_configs))
for idx, cfg in enumerate(sweep_configs, start=1):
    print(idx, cfg)


In [ ]:
# 하이퍼파라미터 sweep 실행
if not corpus or not val_corpus:
    raise RuntimeError("LM train/val corpus가 필요합니다. 데이터 준비 셀을 먼저 실행하세요.")

sweep_train_text = corpus[:SWEEP_CORPUS_LIMIT]
sweep_val_text = val_corpus[:SWEEP_VAL_CORPUS_LIMIT]

sweep_tokenize_started = time.perf_counter()
sweep_train_token_ids = report_tokenizer.encode(sweep_train_text)
sweep_val_token_ids = report_tokenizer.encode(sweep_val_text)
sweep_tokenize_elapsed = time.perf_counter() - sweep_tokenize_started
print("sweep train tokens:", len(sweep_train_token_ids))
print("sweep val tokens:", len(sweep_val_token_ids))
print("sweep tokenization elapsed:", fmt_seconds(sweep_tokenize_elapsed))


def make_sweep_loaders(cfg: dict):
    context_length = int(cfg["context_length"])
    batch_size = int(cfg["batch_size"])
    train_loader = create_dataloader(
        sweep_train_token_ids,
        context_length=context_length,
        batch_size=batch_size,
        stride=context_length,
        drop_last=True,
        shuffle=True,
        num_workers=0,
    )
    val_loader = create_dataloader(
        sweep_val_token_ids,
        context_length=context_length,
        batch_size=batch_size,
        stride=context_length,
        drop_last=False,
        shuffle=False,
        num_workers=0,
    )
    if len(train_loader) == 0 or len(val_loader) == 0:
        raise RuntimeError(f"DataLoader가 비어 있습니다: {cfg}")
    return train_loader, val_loader


def run_sweep_trial(cfg: dict, trial_index: int) -> dict:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model_config = build_sweep_model_config(cfg)
    train_loader, val_loader = make_sweep_loaders(cfg)
    model = GPTModel(model_config).to(REPORT_DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["learning_rate"]),
        weight_decay=SWEEP_WEIGHT_DECAY,
    )

    started = time.perf_counter()
    history = []
    tokens_seen = 0
    global_step = 0
    last_train_loss = None

    model.train()
    while global_step < SWEEP_MAX_STEPS:
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, REPORT_DEVICE)
            loss.backward()
            optimizer.step()

            last_train_loss = loss.item()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % SWEEP_EVAL_FREQ == 0 or global_step >= SWEEP_MAX_STEPS:
                train_loss, val_loss = evaluate_model(
                    model,
                    train_loader,
                    val_loader,
                    REPORT_DEVICE,
                    SWEEP_EVAL_ITER,
                )
                history.append({
                    "step": global_step,
                    "tokens_seen": tokens_seen,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "elapsed_sec": time.perf_counter() - started,
                })
                print(
                    f"[{trial_index}/{len(sweep_configs)}] {cfg.get('name', '')} "
                    f"step={global_step} train_loss={train_loss:.4f} val_loss={val_loss:.4f}"
                )

            if global_step >= SWEEP_MAX_STEPS:
                break

    final_train_loss, final_val_loss = evaluate_model(
        model,
        train_loader,
        val_loader,
        REPORT_DEVICE,
        SWEEP_EVAL_ITER,
    )

    model.eval()
    start_ids = torch.tensor(
        report_tokenizer.encode(SWEEP_START_CONTEXT),
        dtype=torch.long,
        device=REPORT_DEVICE,
    ).unsqueeze(0)
    with torch.no_grad():
        sampled_ids = generate(
            model=model,
            idx=start_ids,
            max_new_tokens=SWEEP_MAX_NEW_TOKENS,
            context_size=model_config["context_length"],
            temperature=0.8,
            top_k=40,
            eos_id=report_tokenizer.get_eos_id(),
        )
    sample_text = safe_decode_token_ids(report_tokenizer, sampled_ids.squeeze(0).tolist())

    result = {
        "trial": trial_index,
        "name": cfg.get("name", f"trial_{trial_index}"),
        **{key: cfg[key] for key in ["batch_size", "drop_rate", "learning_rate", "context_length", "n_layers", "emb_dim"]},
        "steps": global_step,
        "tokens_seen": tokens_seen,
        "last_batch_loss": last_train_loss,
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "best_val_loss": min(row["val_loss"] for row in history) if history else final_val_loss,
        "param_count": count_parameters(model),
        "elapsed_sec": time.perf_counter() - started,
        "sample": sample_text,
        "history": history,
    }

    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

sweep_started = time.perf_counter()
sweep_results = []
for trial_index, cfg in enumerate(sweep_configs, start=1):
    sweep_results.append(run_sweep_trial(cfg, trial_index))

sweep_elapsed = time.perf_counter() - sweep_started
sweep_results_sorted = sorted(sweep_results, key=lambda row: row["final_val_loss"])

sweep_results_path = repo_dir / "checkpoints" / "hparam_sweep_results.json"
sweep_results_path.parent.mkdir(parents=True, exist_ok=True)
with sweep_results_path.open("w", encoding="utf-8") as f:
    json.dump(sweep_results, f, ensure_ascii=False, indent=2)

print("sweep elapsed:", fmt_seconds(sweep_elapsed))
print("best config:", sweep_results_sorted[0])
print("saved:", sweep_results_path)


In [ ]:
# 하이퍼파라미터 sweep 결과 Markdown 표 출력
from IPython.display import Markdown, display


def sweep_markdown_table(results: list[dict]) -> str:
    header = [
        "rank",
        "name",
        "batch_size",
        "drop_rate",
        "lr",
        "context_length",
        "n_layers",
        "emb_dim",
        "params",
        "train_loss",
        "val_loss",
        "best_val",
        "elapsed",
    ]
    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for rank, row in enumerate(sorted(results, key=lambda item: item["final_val_loss"]), start=1):
        lines.append(
            "| "
            + " | ".join([
                str(rank),
                str(row["name"]),
                str(row["batch_size"]),
                str(row["drop_rate"]),
                str(row["learning_rate"]),
                str(row["context_length"]),
                str(row["n_layers"]),
                str(row["emb_dim"]),
                f"{row['param_count']:,}",
                f"{row['final_train_loss']:.4f}",
                f"{row['final_val_loss']:.4f}",
                f"{row['best_val_loss']:.4f}",
                fmt_seconds(row["elapsed_sec"]),
            ])
            + " |"
        )
    return "\n".join(lines)

sweep_markdown = "\n".join([
    "### 하이퍼파라미터 비교 결과",
    "",
    sweep_markdown_table(sweep_results),
    "",
    f"총 실험 수: {len(sweep_results)}",
    f"총 소요 시간: {fmt_seconds(sweep_elapsed)}",
    f"결과 저장 경로: `{sweep_results_path.relative_to(repo_dir)}`",
])

display(Markdown(sweep_markdown))
print(sweep_markdown)

print("\n생성 샘플 preview")
for row in sorted(sweep_results, key=lambda item: item["final_val_loss"])[:3]:
    print(f"[{row['name']}] {compact_text(row['sample'], max_len=240)}")


## 11. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.


In [ ]:
run_pytest("tests/")